# Notebook 03 — Construction du panel final (fusion + préparation pour la modélisation)

**Objectif de ce notebook :**
- **Partie A — Fusion** : assembler les 3 fichiers nettoyés au notebook 02
  (`characteristics_clean.parquet`, `returns_clean.parquet`, `macro_clean.parquet`) en un seul panel,
  avec l'alignement temporel correct pour chaque source, et calculer la variable cible
  (`excess_return = RET - Rfree`)
- **Partie B — Préparation pour la modélisation** : filtrer l'univers investissable (taille,
  liquidité), imputer et winsoriser les caractéristiques restantes, transformer les
  caractéristiques en rangs. Pas de découpage temporel ni de standardisation des variables
  macro ici : les notebooks 04 à 07 utilisent des **fenêtres d'entraînement
  glissantes/extensives** (voir `fenetres.py` et `config.py`) plutôt qu'un split fixe unique,
  donc les deux sont recalculés à la volée pour chaque fenêtre (voir partie B.5)

Ces deux parties sont **séquentielles** (la partie B part directement du panel construit par la
partie A, sans repasser par le disque) — contrairement au notebook 02, où les 3 parties étaient
indépendantes. C'est précisément pour ça qu'elles restent regroupées dans un seul notebook,
avec un seul chargement de données en tête.

⚠️ **Pré-requis : avoir déjà exécuté le notebook 02** (les 3 parties) sur tes fichiers complets.

Partie A sauvegarde `data/processed/panel_final.parquet` (utile comme point de contrôle
intermédiaire, par exemple pour redémarrer uniquement à partir de la partie B plus tard).
Partie B sauvegarde `data/processed/panel_pret_modelisation.parquet`, le fichier consommé par les
3 modèles (notebooks 04 à 06).

> **Note — version optimisee memoire.** Ce notebook reprend `03_construction_panel.ipynb` a l'identique, avec les changements suivants pour reduire l'empreinte memoire :
> - suppression des `.copy()` inutiles sur `panel` (le resultat de `dropna`, `drop_duplicates` ou d'un filtre booleen est deja une nouvelle copie, le `.copy()` en plus doublait temporairement la memoire utilisee) ;
> - `permno` et `annee_mois` sont convertis en `category` juste apres la fusion (partie A), car ces deux colonnes sont repetees des millions de fois dans le panel — puis reconverties dans leur dtype d'origine juste avant la sauvegarde finale (B.7), pour que les notebooks 04 a 08 ne voient aucune difference ;
> - les boucles `for col in caracteristiques: panel.groupby('annee_mois')[col].transform(...)` (imputation, winsorizing, rank transform) sont remplacees par un seul objet `groupby` reutilise pour toutes les colonnes d'un coup, au lieu d'en recreer un a chaque iteration ;
> - `import gc` est deplace dans le bloc d'imports (partie 0) pour etre disponible que la partie A ait tourne ou non dans la session.
>
> La conversion en `float32` n'a volontairement **pas** ete appliquee ici.


## 0. Import des bibliothèques

Un seul bloc d'imports, réutilisé par les deux parties ci-dessous.

In [1]:
import pandas as pd
import numpy as np
import sys
import gc  # importe des le debut pour etre disponible dans toute la session
sys.path.append("..")  # pour pouvoir importer config.py, situe a la racine du projet
import config

config.assurer_dossiers()  # cree data/interim, data/processed, modeles/, outputs/ si absents

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Partie A — Fusion des 3 bases nettoyées

**C'est l'étape la plus délicate du projet :**
1. Charger les 3 fichiers produits par le notebook 02 (parties A, B, C)
2. Créer les clés de fusion en respectant l'**alignement temporel** de chaque source (voir
   A.2 et A.5 — l'alignement n'est **pas le même** pour les caractéristiques/rendements que
   pour les variables macro)
3. Vérifier le chevauchement (entreprises, périodes) **avant** de fusionner
4. Fusionner caractéristiques + rendements sur (`permno`, `annee_mois`)
5. Fusionner le résultat avec les variables macro, en décalant les 8 prédicteurs d'un mois
6. Calculer la variable cible finale : le **rendement excédentaire** (`RET - Rfree`)
7. Sauvegarder le panel dans `data/processed/panel_final.parquet`

ℹ️ **Sur l'alignement temporel — point clé de cette partie :** `datashare.parquet` (Dacheng Xiu)
est **déjà pré-décalé** (« pre-lagged ») dans le fichier source lui-même : son README précise
que pour une ligne `DATE=19570329`, le `RET` de 195703 est directement la bonne variable à
expliquer — les caractéristiques mensuelles y sont décalées d'un mois, les trimestrielles de
4 mois, les annuelles d'un an, précisément pour éviter le look-ahead bias. **Aucun décalage
supplémentaire n'est donc nécessaire entre caractéristiques et rendements** (A.2). En revanche,
les **prédicteurs macro (Welch-Goyal)** de la partie C du notebook 02 ne font pas partie de ce
fichier pré-traité et ne sont donc **pas** pré-décalés : c'est nous qui devons les décaler d'un
mois pour rester cohérents avec le reste (A.5).

### A.1 Chargement des 3 fichiers intermédiaires

In [2]:
chars = pd.read_parquet(config.FICHIER_CARACTERISTIQUES_CLEAN)
# Contrairement au CSV, le Parquet preserve nativement le type de chaque colonne : pas besoin
# de forcer annee_mois en texte ici, elle a deja ete sauvegardee comme telle au notebook 02.
returns = pd.read_parquet(config.FICHIER_RETURNS_CLEAN)
macro = pd.read_parquet(config.FICHIER_MACRO_CLEAN)

print("Caracteristiques :", chars.shape, list(chars.columns))
print("Rendements       :", returns.shape, list(returns.columns))
print("Macro            :", macro.shape, list(macro.columns))


Caracteristiques : (3344804, 64) ['permno', 'DATE', 'annee', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'herf', 'hire', 'lev', 'lgr', 'mve_ia', 'operprof', 'pchgm_pchsale', 'ps', 'quick', 'rd', 'roic', 'salecash', 'salerec', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'secteur']
Rendements       : (4669780, 3) ['permno', 'annee_mois', 'RET']
Macro            : (552, 10) ['annee_mois', 'macro_dp', 'macro_ep', 'macro_bm', 'macro_ntis', 'macro_tbl', 'macro_tms', 'macro_dfy', 'macro_svar', 'Rfree']


### A.2 Créer la clé de fusion mensuelle pour les caractéristiques

`characteristics_clean.parquet` a gardé la colonne `DATE` (format `AAAAMMJJ`) mais pas encore de
colonne `annee_mois`.

ℹ️ **Pas de décalage à faire ici.** `datashare.parquet` est déjà pré-décalé par Xiu : la ligne
datée `DATE` est déjà prévue pour être utilisée directement avec le `RET` du même mois
(ex. `DATE=19570329` → utiliser directement le `RET` de 195703). On se contente donc de
tronquer `DATE` au mois, sans arithmétique supplémentaire.


In [3]:
chars['annee_mois'] = chars['DATE'].astype(str).str[:6]

print("Exemple :")
print(chars[['DATE', 'annee_mois']].head(3))


Exemple :
       DATE annee_mois
0  19800131     198001
1  19800131     198001
2  19800131     198001


### A.3 Vérifier le chevauchement avant de fusionner

C'est l'étape qu'on ne doit **jamais sauter** : si le chevauchement est faible ou nul, la
fusion produira un panel vide ou minuscule, et il vaut mieux le savoir maintenant qu'après
avoir lancé un modèle dessus.

In [4]:
permnos_chars = set(chars['permno'].unique())
permnos_returns = set(returns['permno'].unique())
communs = permnos_chars & permnos_returns

print(f"Entreprises dans characteristics_clean : {len(permnos_chars)}")
print(f"Entreprises dans returns_clean         : {len(permnos_returns)}")
print(f"Entreprises communes                   : {len(communs)}")
print(f"-> {len(communs) / max(len(permnos_chars), 1) * 100:.1f}% des entreprises de chars "
      f"ont au moins un rendement correspondant")


Entreprises dans characteristics_clean : 29731
Entreprises dans returns_clean         : 38297
Entreprises communes                   : 29709
-> 99.9% des entreprises de chars ont au moins un rendement correspondant


In [5]:
periodes_chars = set(chars['annee_mois'])
periodes_returns = set(returns['annee_mois'])
periodes_macro = set(macro['annee_mois'])

print(f"Periodes chars   : {len(periodes_chars)} mois "
      f"({min(periodes_chars, default='-')} a {max(periodes_chars, default='-')})")
print(f"Periodes returns : {len(periodes_returns)} mois "
      f"({min(periodes_returns, default='-')} a {max(periodes_returns, default='-')})")
print(f"Periodes macro   : {len(periodes_macro)} mois "
      f"({min(periodes_macro, default='-')} a {max(periodes_macro, default='-')})")


Periodes chars   : 504 mois (198001 a 202112)
Periodes returns : 828 mois (195601 a 202412)
Periodes macro   : 552 mois (198001 a 202512)


**Si le chevauchement te semble faible sur ton fichier complet**, la cause la plus
fréquente est un décalage de `ANNEE_DEBUT` entre les notebooks 02 et 04, ou un identifiant
d'entreprise incompatible entre `datashare.parquet` et `StockReturn.parquet` (vérifie qu'il s'agit
bien du même type de code `permno`/`PERMNO` dans les deux fichiers d'origine).

### A.4 Fusion 1 — Caractéristiques + Rendements

On utilise une fusion **inner** (intersection) sur (`permno`, `annee_mois`) : on ne garde
que les lignes où on a **à la fois** les caractéristiques d'une entreprise pour un mois donné
**et** son rendement pour ce même mois. Une ligne sans l'un des deux n'est pas utilisable
pour entraîner un modèle supervisé.

In [6]:
avant_chars, avant_returns = len(chars), len(returns)

panel = pd.merge(
    chars, returns,
    on=['permno', 'annee_mois'],
    how='inner',
    validate='one_to_one',  # leve une erreur si des doublons de cle existent d'un cote ou l'autre
)

print(f"Lignes characteristics avant fusion : {avant_chars}")
print(f"Lignes returns avant fusion         : {avant_returns}")
print(f"Lignes apres fusion (intersection)  : {len(panel)}")


Lignes characteristics avant fusion : 3344804
Lignes returns avant fusion         : 4669780
Lignes apres fusion (intersection)  : 3326704


ℹ️ Si la cellule ci-dessus lève une erreur `MergeError` mentionnant des doublons, c'est
qu'il reste des lignes en double sur (`permno`, `annee_mois`) dans un des deux fichiers —
retourne à la partie A ou B correspondante du notebook 02 pour les identifier et les
traiter avant de
continuer.

### A.5 Fusion 2 — Ajout des variables macro

Ici, contrairement à la section 2, il **faut** décaler — mais seulement une partie des
variables macro, et pas de la même façon :
- les **8 prédicteurs macro** (`macro_dp`, `macro_ep`, ...) viennent de `MacroData.parquet`
  (Welch-Goyal), qui n'est **pas** pré-décalé comme `datashare.parquet`. Pour qu'un prédicteur
  macro connu au mois *m* serve bien à prédire le rendement du mois suivant (et non à
  « voir » un mois macro qui n'était pas encore connu), on le décale nous-mêmes d'un mois :
  le prédicteur du mois *m* est fusionné sur la ligne dont `annee_mois` vaut *m+1*.
- `Rfree`, lui, sert à calculer le **rendement excédentaire du même mois** que `RET`
  (`excess_return = RET - Rfree`, section 6) → **pas de décalage** : on le fusionne
  directement sur le même `annee_mois` que `RET`.

On fait donc 2 fusions successives, chacune en `left` (on garde toutes les lignes du panel),
puis on vérifie explicitement si des mois n'ont pas trouvé de correspondance macro (ce qui
arrivera forcément pour le tout premier mois du panel : il n'existe pas de prédicteur macro
pour le mois précédent `ANNEE_DEBUT`).

In [7]:
def mois_suivant(annee_mois):
    """Renvoie le mois suivant au format AAAAMM (texte).

    Sert a decaler les predicteurs macro (Welch-Goyal, PAS pre-decales contrairement
    a datashare.parquet) : un predicteur connu au mois m doit servir a predire le
    rendement du mois m+1, jamais celui du mois m lui-meme.
    """
    annee = int(annee_mois[:4])
    mois = int(annee_mois[4:6])
    mois += 1
    if mois > 12:
        mois = 1
        annee += 1
    return f"{annee:04d}{mois:02d}"


# 1) Les 8 predicteurs macro : le predicteur connu au mois m est decale vers m+1,
#    pour etre fusionne avec la ligne du panel dont annee_mois == m+1.
macro_predicteurs_decales = macro[['annee_mois'] + config.MACRO_PREDICTEURS].copy()
macro_predicteurs_decales['annee_mois'] = macro_predicteurs_decales['annee_mois'].apply(mois_suivant)

avant_fusion_predicteurs = len(panel)
panel = pd.merge(panel, macro_predicteurs_decales, on='annee_mois', how='left')

print(f"Lignes avant fusion predicteurs macro : {avant_fusion_predicteurs}")
print(f"Lignes apres fusion predicteurs macro : {len(panel)}")

# 2) Rfree : PAS de decalage, il correspond au meme mois que RET (le rendement
#    excedentaire se calcule sur la meme periode, section 6).
rfree_avec_cle = macro[['annee_mois', 'Rfree']]

avant_fusion_rfree = len(panel)
panel = pd.merge(panel, rfree_avec_cle, on='annee_mois', how='left')

print(f"Lignes avant fusion Rfree : {avant_fusion_rfree}")
print(f"Lignes apres fusion Rfree : {len(panel)}")


Lignes avant fusion predicteurs macro : 3326704
Lignes apres fusion predicteurs macro : 3326704
Lignes avant fusion Rfree : 3326704
Lignes apres fusion Rfree : 3326704


In [8]:
# 1. Libération de la mémoire inutilisée des étapes précédentes (gc importé en partie 0)
del chars, returns
gc.collect()

colonnes_macro_predicteurs = config.MACRO_PREDICTEURS + ['Rfree']
manquant_macro = panel[colonnes_macro_predicteurs].isna().any(axis=1).sum()

print(f"Lignes sans donnees macro correspondantes : {manquant_macro} "
      f"({manquant_macro / len(panel) * 100:.2f}%)")

if manquant_macro > 0:
    print()
    print("Ces lignes sont probablement dues a un decalage de periode entre chars/returns")
    print("et macro (verifie ANNEE_DEBUT dans config.py). On les retire pour garder un panel final complet.")
    panel = panel.dropna(subset=colonnes_macro_predicteurs)
    print(f"Lignes apres suppression : {len(panel)}")


Lignes sans donnees macro correspondantes : 4764 (0.14%)

Ces lignes sont probablement dues a un decalage de periode entre chars/returns
et macro (verifie ANNEE_DEBUT dans config.py). On les retire pour garder un panel final complet.
Lignes apres suppression : 3321940


Suivant l'approche de Gu, Kelly & Xiu (2020), la variable à prédire n'est pas le rendement
brut `RET` mais le **rendement excédentaire** par rapport au taux sans risque :
`excess_return = RET - Rfree`. Grâce à la fusion de la section 5 (`Rfree` non décalé,
contrairement aux 8 prédicteurs macro), `RET` et `Rfree` correspondent bien tous les deux
au même mois.

In [9]:
panel['excess_return'] = panel['RET'] - panel['Rfree']

panel[['permno', 'annee_mois', 'RET', 'Rfree', 'excess_return']].head()

,permno,annee_mois,RET,Rfree,excess_return
4764,10006,198002,-0.058795,0.0089,-0.067695
4765,10057,198002,-0.183582,0.0089,-0.192482
4766,10058,198002,0.000000,0.0089,-0.008900
4767,10065,198002,-0.029126,0.0089,-0.038026
4768,10103,198002,-0.230769,0.0089,-0.239669


In [10]:
del macro, macro_predicteurs_decales, rfree_avec_cle
del permnos_chars, permnos_returns, communs, periodes_chars, periodes_returns, periodes_macro
gc.collect()

0

### A.6 Vérifications finales

In [11]:
doublons = panel.duplicated(subset=['permno', 'annee_mois']).sum()
print(f"Doublons (permno, annee_mois) restants : {doublons}")
if doublons > 0:
    panel = panel.drop_duplicates(subset=['permno', 'annee_mois'], keep='first')
    print("Doublons retires (on garde la premiere occurrence).")


Doublons (permno, annee_mois) restants : 0


In [12]:
print("Dimensions finales du panel :", panel.shape)
print("Nombre d'entreprises uniques :", panel['permno'].nunique())
print("Periode couverte : de", panel['annee_mois'].min(), "a", panel['annee_mois'].max())
print()
print("Valeurs manquantes restantes par colonne (doit etre 0 partout) :")
manquants = panel.isna().sum()
print(manquants[manquants > 0] if manquants.sum() > 0 else "Aucune valeur manquante.")


Dimensions finales du panel : (3321940, 76)
Nombre d'entreprises uniques : 29657
Periode couverte : de 198002 a 202112

Valeurs manquantes restantes par colonne (doit etre 0 partout) :
mvel1              1097
beta             314221
betasq           314221
chmom            265103
dolvol           139964
idiovol          314221
indmom                9
mom1m             24733
mom6m            122705
mom12m           265103
mom36m           761046
pricedelay       314293
turn             142005
age              688194
agr              888725
bm               709454
bm_ia            709454
cashdebt         817812
cashpr           718526
cfp              875755
cfp_ia           875755
chcsho           892316
chempia          928854
chinv            949510
chpmia           938016
convind          688194
currat           794586
depr             880234
divi             888682
divo             888682
dy               701918
egr              890863
ep               691178
gma              893702

### A.6bis Optimisation mémoire — `annee_mois` et `permno` en `category`

Ces deux colonnes ne prennent qu'un petit nombre de valeurs distinctes (le nombre de mois, le nombre d'entreprises) mais chacune de ces valeurs est répétée sur des millions de lignes. Le type `category` ne stocke chaque valeur distincte qu'une seule fois en interne (des entiers servent ensuite de références), ce qui réduit nettement l'empreinte mémoire du panel sans rien changer à son contenu. On fait cette conversion ici, une fois pour toutes, juste avant la sauvegarde : elle sera donc aussi présente si la partie B recharge `panel_final.parquet` depuis le disque plutôt que de réutiliser `panel` en mémoire (cf. B.0 ci-dessous).


In [13]:
# annee_mois a un ordre naturel (chronologique) : on cree la category en 'ordered=True'
# pour pouvoir continuer a utiliser .min() / .max() dessus (ex. section B.7 plus bas).
# permno est un simple identifiant, aucun ordre n'est necessaire.
panel['annee_mois'] = panel['annee_mois'].astype('category').cat.as_ordered()
panel['permno'] = panel['permno'].astype('category')

print("Types apres conversion :")
print(panel[['permno', 'annee_mois']].dtypes)
print("\nMemoire utilisee par le panel (en Mo) :", panel.memory_usage(deep=True).sum() / 1e6)


Types apres conversion :
permno        category
annee_mois    category
dtype: object

Memoire utilisee par le panel (en Mo) : 1987.771955


In [14]:
panel['excess_return'].describe()


count    3.321940e+06
mean     7.457279e-03
std      1.822209e-01
min     -9.952000e-01
25%     -6.510000e-02
50%     -1.785000e-03
75%      6.276700e-02
max      2.399660e+01
Name: excess_return, dtype: float64

### A.7 Sauvegarde du panel final

In [15]:
panel.to_parquet(config.FICHIER_PANEL_FINAL, index=False)
print("Fichier sauvegarde :", config.FICHIER_PANEL_FINAL)
print(f"Taille du fichier final : {panel.shape[0]} lignes x {panel.shape[1]} colonnes")


Fichier sauvegarde : C:\Users\aless\OneDrive\Documents\Pantheon Sorbonne\cour\Memoire\V11\data\processed\panel_final.parquet
Taille du fichier final : 3321940 lignes x 76 colonnes


**Résumé de la partie A :** fusion caractéristiques + rendements en `inner` sur
(`permno`, `annee_mois`), fusion avec les variables macro en deux temps (8 prédicteurs décalés
d'un mois, `Rfree` non décalé), cible finale `excess_return = RET - Rfree`. Panel sauvegardé
dans `data/processed/panel_final.parquet`.

On enchaîne directement avec la partie B ci-dessous, **sans recharger `panel_final.parquet` depuis
le disque** : la variable `panel` en mémoire est déjà celle dont on a besoin. (Si tu veux un
jour repartir uniquement de la partie B dans une exécution future, tu peux recharger ce fichier
avec `pd.read_parquet(config.FICHIER_PANEL_FINAL)`.)

## Partie B — Préparation pour la modélisation

1. Filtrer l'univers investissable : taille (garder les 2 quartiles supérieurs de
   capitalisation boursière `mvel1`, chaque mois) puis liquidité (retirer les titres les plus
   illiquides selon `ill`, critère d'Amihud, chaque mois)
2. Imputer les valeurs manquantes restantes des caractéristiques retenues hors `mvel1`
   (déplacé ici depuis le notebook 02, partie A — voir B.4)
3. Winsoriser les caractéristiques (déplacé ici depuis le notebook 02, partie A — voir B.5)
4. Découper temporellement en train / validation / test
5. Transformer les caractéristiques en rangs (rank transform), calculé après les filtres
   taille + liquidité pour que chaque rang reflète la position dans l'univers réellement
   modélisé
6. Standardiser les variables macro (moyenne/écart-type calculés sur `train` uniquement)
7. Sauvegarder `panel_pret_modelisation.parquet`, consommé par les notebooks 04, 05 et 06

ℹ️ **Pourquoi autant d'opérations sont faites ici plutôt qu'au notebook 02 (partie A) ?**
Toutes les statistiques utilisées pour nettoyer les caractéristiques (médiane d'imputation,
seuils de winsorizing, rangs) doivent être calculées sur la **même population** que celle qui
sera réellement modélisée. Si on les calcule avant les filtres taille + liquidité, elles
seraient influencées par des entreprises (micro-caps, titres illiquides) qu'on exclut de toute
façon ensuite — ce n'est pas une fuite de données au sens temporel (chaque calcul reste fait
mois par mois, jamais entre périodes), mais c'est statistiquement plus cohérent de filtrer
l'univers d'abord.

### B.0 Point de départ : le panel fusionné

Si tu viens d'exécuter la partie A dans **cette même session** de kernel, le DataFrame
`panel` est déjà en mémoire et la cellule ci-dessous ne fait que l'afficher. Mais si tu
redémarres le kernel pour ne relancer **que la partie B** (par exemple pour tester un autre
seuil de filtrage sur `mvel1`, en B.2.1), `panel` n'existe plus : la cellule le recharge alors
automatiquement depuis `data/processed/panel_final.parquet`, sauvegardé à la fin de la partie A.

👉 **En pratique, pour tester un nouveau seuil de filtrage (ou tout autre changement limité à
la partie B) :** pas besoin de relancer la partie A. Redémarre le kernel (`Kernel > Restart`),
exécute la cellule d'imports (section 0) puis toutes les cellules de la partie B (`Run > Run
Selected Cell and All Below` depuis B.0) — la cellule ci-dessous rechargera `panel_final.parquet`
automatiquement, sans repasser par la fusion.

In [16]:
try:
    panel
    print("Panel deja en memoire (partie A executee dans cette session), shape :", panel.shape)
except NameError:
    panel = pd.read_parquet(config.FICHIER_PANEL_FINAL)
    print("Panel recharge depuis le disque (partie A non executee dans cette session), "
          f"shape : {panel.shape}")

print("Colonnes :", list(panel.columns))
panel.head()

Panel deja en memoire (partie A executee dans cette session), shape : (3321940, 76)
Colonnes : ['permno', 'DATE', 'annee', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'herf', 'hire', 'lev', 'lgr', 'mve_ia', 'operprof', 'pchgm_pchsale', 'ps', 'quick', 'rd', 'roic', 'salecash', 'salerec', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'secteur', 'annee_mois', 'RET', 'macro_dp', 'macro_ep', 'macro_bm', 'macro_ntis', 'macro_tbl', 'macro_tms', 'macro_dfy', 'macro_svar', 'Rfree', 'excess_return']


,permno,DATE,annee,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,...,macro_dp,macro_ep,macro_bm,macro_ntis,macro_tbl,macro_tms,macro_dfy,macro_svar,Rfree,excess_return
4764,10006,19800229,1980,367648.50,1.059229,1.121966,-0.004264,10.985509,0.025585,0.172000,...,-2.997135,-2.029331,1.016955,0.011297,0.12,-0.0086,0.0133,0.001906,0.0089,-0.067695
4765,10057,19800229,1980,142408.50,1.496519,2.239569,0.717354,10.579441,0.036105,0.387264,...,-2.997135,-2.029331,1.016955,0.011297,0.12,-0.0086,0.0133,0.001906,0.0089,-0.192482
4766,10058,19800229,1980,3127.50,0.499751,0.249751,0.250000,NaN,0.088678,0.320007,...,-2.997135,-2.029331,1.016955,0.011297,0.12,-0.0086,0.0133,0.001906,0.0089,-0.008900
4767,10065,19800229,1980,201159.00,0.558640,0.312079,0.040972,9.554036,0.019289,0.320007,...,-2.997135,-2.029331,1.016955,0.011297,0.12,-0.0086,0.0133,0.001906,0.0089,-0.038026
4768,10103,19800229,1980,2213.25,1.736435,3.015207,0.181818,NaN,0.090373,0.286108,...,-2.997135,-2.029331,1.016955,0.011297,0.12,-0.0086,0.0133,0.001906,0.0089,-0.239669


### B.1 Définir les groupes de colonnes

On distingue clairement 4 rôles : les identifiants, les caractéristiques d'entreprise (à
rank-transformer), les variables macro (à standardiser), et la variable cible.

Ces listes viennent maintenant de `config.py` — si tu modifies le seuil de valeurs
manquantes (`SEUIL_MAX_PCT_MANQUANT_CARACTERISTIQUES`) ou la liste candidate au notebook 02
(partie A, section A.3bis), cette partie B (et les notebooks 04 à 06) se mettent à jour
automatiquement, sans rien recopier à la main — `config.CARACTERISTIQUES_RETENUES` lit
directement le manifeste que le notebook 02 a sauvegardé, `characteristics_clean.parquet`
ne contenant de toute façon déjà plus que ces colonnes-là.

In [17]:
caracteristiques = config.CARACTERISTIQUES_RETENUES
macro_predicteurs = config.MACRO_PREDICTEURS
cible = config.CIBLE

# Verification que toutes ces colonnes existent bien dans le panel
colonnes_attendues = caracteristiques + macro_predicteurs + [cible]
colonnes_manquantes = [c for c in colonnes_attendues if c not in panel.columns]
if colonnes_manquantes:
    print("ATTENTION, colonnes introuvables dans le panel :", colonnes_manquantes)
else:
    print(f"OK : les {len(caracteristiques)} caracteristiques retenues + {len(macro_predicteurs)} "
          "predicteurs macro sont presents dans le panel.")

OK : les 60 caracteristiques retenues + 8 predicteurs macro sont presents dans le panel.


### B.2 Filtrage de l'univers investissable (taille + liquidité)

On applique maintenant deux filtres successifs, mois par mois, avant toute imputation ou
winsorisation : d'abord sur la **taille** (`mvel1`), puis sur la **liquidité** (`ill`).
Les deux visent le même objectif — retirer les titres les plus sujets au bruit de
microstructure — mais capturent des dimensions différentes : une entreprise peut être de
taille moyenne tout en étant peu échangée (et inversement).

#### B.2.1 Filtrage sur la taille (capitalisation boursière `mvel1`)

On n'a pas de colonne `PRC` (prix) pour appliquer le filtre standard de la littérature
(exclure les titres à prix très bas, sujets à un bruit de microstructure important —
spreads bid-ask énormes en proportion du prix, mouvements en % démesurés pour des
variations en dollars négligeables). On utilise `mvel1` (capitalisation boursière, déjà
dans nos caractéristiques retenues) comme proxy.

**Le filtre est appliqué mois par mois** (jamais sur une moyenne de vie entière d'une
entreprise) : c'est la pratique standard (Fama-French, Hou-Xue-Zhang) car (1) une
moyenne calculée sur toute la période inclurait de l'information du futur pour une
décision qui affecte des lignes passées — une fuite de données — et (2) ça exclurait
*toute* une entreprise (y compris ses mois où elle était grande et fiable) juste parce
que sa moyenne globale est petite. Le filtre mensuel est plus chirurgical : une ligne
est exclue seulement pour les mois où l'entreprise était vraiment petite.

🔧 **Consigne du professeur : filtrage sur la taille.** On ne garde que les **2 quartiles
supérieurs** de `mvel1` chaque mois, c'est-à-dire qu'on exclut la **moitié inférieure**
(les 2 quartiles les plus petits) de l'univers.

✅ **Le seuil est dans `config.py`** (`SEUIL_PERCENTILE_TAILLE = 0.50`) -- change-le
là-bas, pas ici. Il correspond à la consigne « 2 quartiles inférieurs exclus » décrite
ci-dessus.

⚠️ **Ces deux filtres (taille, puis liquidité juste en dessous) doivent passer avant
l'imputation (section 4) et le winsorizing (section 5)** : la winsorisation capperait par
exemple les valeurs les plus basses de `mvel1` à son 1er centile, ce qui "remonterait"
artificiellement les plus petites capitalisations et fausserait le calcul du seuil
ci-dessous. C'est pourquoi le winsorizing des caractéristiques se fait ici, dans ce
notebook, après ces deux filtres.

ℹ️ **Note — traitement des `mvel1` manquants** :
`mvel1` n'est jamais imputé (fabriquer une taille empêcherait le filtre de fonctionner
pour ces lignes). Ses valeurs manquantes sont retirées juste ci-dessous, au même endroit
que le filtre lui-même — exactement comme pour `ill` en B.2.2.

In [18]:
# Etape 1 : on retire les lignes ou 'mvel1' est manquant -- regroupe ici avec le
# nettoyage de 'ill' (B.2.2), les deux caracteristiques qui servent de critere de filtre.
# Tres peu de lignes concernees.
avant_dropna_mvel1 = len(panel)
entreprises_avant_dropna_mvel1 = panel['permno'].nunique()

panel = panel.dropna(subset=['mvel1'])

apres_dropna_mvel1 = len(panel)
entreprises_apres_dropna_mvel1 = panel['permno'].nunique()

print(f"Lignes avant suppression des mvel1 manquants : {avant_dropna_mvel1}")
print(f"Lignes apres suppression des mvel1 manquants : {apres_dropna_mvel1} "
      f"({(avant_dropna_mvel1 - apres_dropna_mvel1) / avant_dropna_mvel1 * 100:.2f}% retirees)")
print(f"Entreprises avant : {entreprises_avant_dropna_mvel1}")
print(f"Entreprises apres (au moins 1 mois restant) : {entreprises_apres_dropna_mvel1}")
print()

# Etape 2 : filtre de taille proprement dit, sur la population sans mvel1 manquant.
# Consigne du professeur : on ne garde que les 2 quartiles superieurs de mvel1 chaque
# mois (on exclut la moitie inferieure).
# Parametre GENERAL dans config.py (voir sa docstring) : change-le LA-BAS, pas ici.
SEUIL_PERCENTILE_TAILLE = config.SEUIL_PERCENTILE_TAILLE

seuils_taille_mensuels = panel.groupby('annee_mois')['mvel1'].transform(
    lambda x: x.quantile(SEUIL_PERCENTILE_TAILLE)
)

avant_filtre_taille = len(panel)
entreprises_avant = panel['permno'].nunique()

masque_micro_cap = panel['mvel1'] < seuils_taille_mensuels
panel = panel[~masque_micro_cap]
del seuils_taille_mensuels, masque_micro_cap
gc.collect()

apres_filtre_taille = len(panel)
entreprises_apres = panel['permno'].nunique()

print(f"Lignes avant filtre taille (hors mvel1 manquants) : {avant_filtre_taille}")
print(f"Lignes apres filtre taille : {apres_filtre_taille} "
      f"({(avant_filtre_taille - apres_filtre_taille) / avant_filtre_taille * 100:.2f}% retirees)")
print(f"Entreprises avant (permno distincts) : {entreprises_avant}")
print(f"Entreprises apres (au moins 1 mois restant) : {entreprises_apres}")

Lignes avant suppression des mvel1 manquants : 3321940
Lignes apres suppression des mvel1 manquants : 3320843 (0.03% retirees)
Entreprises avant : 29657
Entreprises apres (au moins 1 mois restant) : 29652

Lignes avant filtre taille (hors mvel1 manquants) : 3320843
Lignes apres filtre taille : 2988598 (10.00% retirees)
Entreprises avant (permno distincts) : 29652
Entreprises apres (au moins 1 mois restant) : 28664


#### B.2.2 Filtrage sur la liquidité (`ill` — critère d'illiquidité d'Amihud)

Deuxième filtre demandé par le professeur, appliqué **après** le filtre de taille
(3.1) et **sur la population déjà réduite** par celui-ci : on retire maintenant les titres
les plus **illiquides** au sens de la mesure d'Amihud (`ill`), déjà présente parmi nos 25
caractéristiques (catégorie "liquidité"). Plus `ill` est élevé, plus le titre est illiquide
(un même volume de transactions y déplace davantage le prix) — on exclut donc les valeurs
les **plus hautes** de `ill` chaque mois, à l'inverse du filtre de taille qui excluait les
valeurs les plus **basses** de `mvel1`.

**Même logique mensuelle que le filtre de taille (B.2.1)**, pour les mêmes raisons (éviter la fuite
de données d'une moyenne calculée sur toute la période, et ne retirer une entreprise que
pour les mois où elle était vraiment illiquide plutôt que toute son historique).

ℹ️ **Note — traitement des `ill` manquants, en cohérence avec `mvel1`** : `mvel1` n'a plus
aucune valeur manquante à ce stade car ces lignes viennent d'être retirées juste au-dessus
(3.1), avant même d'être utilisées comme critère de filtre. On applique le même principe à
`ill` : puisqu'on l'utilise maintenant comme critère de filtre (et plus seulement comme
caractéristique à imputer), on retire d'abord les lignes où `ill` est manquant — une bonne
pratique générale (on ne filtre pas sur une valeur inconnue), et ici peu coûteuse : après le
filtre de taille, il ne reste qu'environ 1,79 % de `ill` manquant. Les 23 autres
caractéristiques, elles, ne servent à aucun filtre : elles restent donc simplement imputées
à la section 4, comme avant.

⚠️ **Seuil à ajuster toi-même** : le professeur a demandé un filtrage sur `ill`, mais sans
préciser de seuil exact. Le paramètre exclut par défaut le **décile le plus
illiquide** de chaque mois (10 %) — ajuste `SEUIL_PERCENTILE_LIQUIDITE` **dans `config.py`**
(pas ici, ce notebook le lit depuis `config.py` — voir aussi la note plus haut sur
`SEUIL_PERCENTILE_TAILLE`) si tu as une consigne plus précise (ex: `0.75` pour exclure le
quart le plus illiquide, en cohérence avec le filtrage par quartiles utilisé pour la taille).

ℹ️ Ces deux seuils sont des **paramètres GÉNÉRAUX** au sens du notebook 08 : ils
sont enregistrés avec chaque expérience dans `outputs/journal_experiences.parquet` (via
`journal.py`), pour garder une trace de l'univers investissable utilisé par chaque modèle
entraîné — voir le notebook 08 pour la comparaison.

In [19]:
# ============================================================
# Parametre GENERAL dans config.py (voir sa docstring) : change-le LA-BAS,
# pas ici, si ton prof te donne un seuil precis pour la liquidite.
SEUIL_PERCENTILE_LIQUIDITE = config.SEUIL_PERCENTILE_LIQUIDITE
# ============================================================

# Etape 1 : on retire les lignes ou 'ill' est manquant -- meme logique que mvel1 en
# B.2.1, puisqu'on utilise 'ill' comme critere de filtre (pas seulement
# comme caracteristique a imputer). Tres peu de lignes concernees a ce stade.
avant_dropna_ill = len(panel)
entreprises_avant_dropna_ill = panel['permno'].nunique()

panel = panel.dropna(subset=['ill'])

apres_dropna_ill = len(panel)
entreprises_apres_dropna_ill = panel['permno'].nunique()

print(f"Lignes avant suppression des ill manquants : {avant_dropna_ill}")
print(f"Lignes apres suppression des ill manquants : {apres_dropna_ill} "
      f"({(avant_dropna_ill - apres_dropna_ill) / avant_dropna_ill * 100:.2f}% retirees)")
print(f"Entreprises avant : {entreprises_avant_dropna_ill}")
print(f"Entreprises apres (au moins 1 mois restant) : {entreprises_apres_dropna_ill}")
print()

# Etape 2 : filtre de liquidite proprement dit, sur la population sans ill manquant
seuils_liquidite_mensuels = panel.groupby('annee_mois')['ill'].transform(
    lambda x: x.quantile(SEUIL_PERCENTILE_LIQUIDITE)
)

avant_filtre_liquidite = len(panel)
entreprises_avant_liquidite = panel['permno'].nunique()

masque_illiquide = panel['ill'] > seuils_liquidite_mensuels
panel = panel[~masque_illiquide]
del seuils_liquidite_mensuels, masque_illiquide
gc.collect()

apres_filtre_liquidite = len(panel)
entreprises_apres_liquidite = panel['permno'].nunique()

print(f"Lignes avant filtre liquidite (hors ill manquants) : {avant_filtre_liquidite}")
print(f"Lignes apres filtre liquidite : {apres_filtre_liquidite} "
      f"({(avant_filtre_liquidite - apres_filtre_liquidite) / avant_filtre_liquidite * 100:.2f}% retirees)")
print(f"Entreprises avant (permno distincts) : {entreprises_avant_liquidite}")
print(f"Entreprises apres (au moins 1 mois restant) : {entreprises_apres_liquidite}")
print()
print("Effet cumule de la section 3 dans son ensemble (taille, puis ill manquants retires,")
print("puis filtre de liquidite), depuis le panel de depart :")
print(f"  Lignes avant (tout debut de la section 3, avant tout nettoyage) : {avant_dropna_mvel1}")
print(f"  Lignes apres (mvel1 et ill manquants retires, taille ET liquidite filtrees) : "
      f"{apres_filtre_liquidite} "
      f"({(avant_dropna_mvel1 - apres_filtre_liquidite) / avant_dropna_mvel1 * 100:.2f}% retirees au total)")

Lignes avant suppression des ill manquants : 2988598
Lignes apres suppression des ill manquants : 2906647 (2.74% retirees)
Entreprises avant : 28664
Entreprises apres (au moins 1 mois restant) : 28087

Lignes avant filtre liquidite (hors ill manquants) : 2906647
Lignes apres filtre liquidite : 2615807 (10.01% retirees)
Entreprises avant (permno distincts) : 28087
Entreprises apres (au moins 1 mois restant) : 27556

Effet cumule de la section 3 dans son ensemble (taille, puis ill manquants retires,
puis filtre de liquidite), depuis le panel de depart :
  Lignes avant (tout debut de la section 3, avant tout nettoyage) : 3321940
  Lignes apres (mvel1 et ill manquants retires, taille ET liquidite filtrees) : 2615807 (21.26% retirees au total)


#### Note de diagnostic (bloc laissé en commentaire, voir ci-dessous)

**Effet des filtres (taille + liquidité) sur les valeurs extrêmes de la cible** — on
compare les statistiques de `excess_return` avant/après, pour voir si les deux filtres
de la section 3 suffisent à éliminer les rendements aberrants (ex. les valeurs proches de
+2500 % identifiées plus tôt dans le projet), ou s'il en reste malgré tout.

In [20]:
"""print("Quantiles extremes de excess_return (apres filtres taille + liquidite) :")
print(panel[cible].quantile([0.001, 0.01, 0.5, 0.99, 0.999]))
print()

seuil_diagnostic = 10  # +500%, meme seuil de diagnostic qu'au notebook 03
nb_extremes_restants = (panel[cible].abs() > seuil_diagnostic).sum()
print(f"Lignes avec |excess_return| > {seuil_diagnostic*100:.0f}% restantes : {nb_extremes_restants}")

if nb_extremes_restants > 0:
    print()
    print("Il reste des valeurs extremes malgre les filtres taille + liquidite : ce sont donc de")
    print("grandes entreprises avec un rendement aberrant (probablement une erreur de")
    print("donnees plutot qu'un effet penny stock). On applique alors, en dernier")
    print("recours, la meme technique de troncature mensuelle qu'au notebook 03 --")
    print("directement sur excess_return cette fois, puisque c'est la variable cible")
    print("finale utilisee par les modeles.")

    SEUIL_PERCENTILE_CIBLE = 0.999  # ne retire que le 0.1% le plus extreme de chaque mois

    seuils_cible_mensuels = panel.groupby('annee_mois')[cible].transform(
        lambda x: x.abs().quantile(SEUIL_PERCENTILE_CIBLE)
    )

    avant_troncature_cible = len(panel)
    masque_extreme_cible = panel[cible].abs() > seuils_cible_mensuels
    panel = panel[~masque_extreme_cible].copy()
    apres_troncature_cible = len(panel)

    print(f"Lignes retirees (troncature residuelle sur {cible}) : "
          f"{avant_troncature_cible - apres_troncature_cible}")
else:
    print("Aucune valeur residuelle extreme : les filtres taille + liquidite suffisent, pas besoin")
    print("de troncature supplementaire sur la cible.")"""


'print("Quantiles extremes de excess_return (apres filtres taille + liquidite) :")\nprint(panel[cible].quantile([0.001, 0.01, 0.5, 0.99, 0.999]))\nprint()\n\nseuil_diagnostic = 10  # +500%, meme seuil de diagnostic qu\'au notebook 03\nnb_extremes_restants = (panel[cible].abs() > seuil_diagnostic).sum()\nprint(f"Lignes avec |excess_return| > {seuil_diagnostic*100:.0f}% restantes : {nb_extremes_restants}")\n\nif nb_extremes_restants > 0:\n    print()\n    print("Il reste des valeurs extremes malgre les filtres taille + liquidite : ce sont donc de")\n    print("grandes entreprises avec un rendement aberrant (probablement une erreur de")\n    print("donnees plutot qu\'un effet penny stock). On applique alors, en dernier")\n    print("recours, la meme technique de troncature mensuelle qu\'au notebook 03 --")\n    print("directement sur excess_return cette fois, puisque c\'est la variable cible")\n    print("finale utilisee par les modeles.")\n\n    SEUIL_PERCENTILE_CIBLE = 0.999  # ne retir

In [21]:
print("% de valeurs manquantes par caracteristique (donnee brute) :")
print((panel[caracteristiques].isna().mean() * 100).sort_values(ascending=False))

% de valeurs manquantes par caracteristique (donnee brute) :
tb               29.660942
operprof         28.866847
chinv            28.148827
chpmia           27.444532
chempia          27.398160
hire             27.398160
pchgm_pchsale    27.317077
sgr              27.306487
depr             26.826291
lgr              26.440253
gma              26.348504
chcsho           26.269866
cfp_ia           26.246508
cfp              26.246508
egr              26.237371
agr              26.171961
divi             26.170853
divo             26.170853
ps               26.170853
rd               26.170853
tang             25.794105
cashdebt         24.949776
quick            24.366706
currat           23.883757
salerec          23.769682
roic             23.511788
mom36m           21.761812
cashpr           21.435450
salecash         21.183673
bm_ia            21.074873
bm               21.074873
dy               20.976471
sp               20.820879
lev              20.819923
ep               20.6

### B.3 Imputation des valeurs manquantes restantes

Les autres caractéristiques autres que `mvel1` et `ill`  peuvent encore contenir des valeurs manquantes à
ce stade — elles n'ont volontairement pas été imputées au notebook 02, partie A (voir son résumé).
On les impute maintenant, **après les filtres taille + liquidité** : la médiane utilisée
pour chaque mois doit venir de la population qu'on garde réellement pour la modélisation,
pas d'une population plus large qui inclurait des micro-caps ou des titres illiquides
qu'on vient d'exclure.

Même logique d'imputation qu'au notebook 02 (partie A) à l'origine (médiane du mois, avec la médiane
globale de la colonne en filet de sécurité si un mois entier est vide, puis 0 en tout dernier
recours si une caractéristique est entièrement vide sur toute la période) — sauf que `mvel1`
n'est jamais concerné ici : il est déjà garanti complet, ses lignes manquantes ayant été
retirées juste au-dessus (section 3.1).


In [22]:
caracteristiques_a_imputer = [c for c in caracteristiques if c != 'mvel1']

def imputer_par_mediane_mois(serie):
    return serie.fillna(serie.median())

# Un seul objet groupby cree et reutilise pour toutes les caracteristiques d'un coup, au lieu
# d'un groupby('annee_mois') refait a chaque iteration d'une boucle sur les colonnes : le
# decoupage par mois n'est calcule qu'une fois, ce qui limite le pic memoire et le travail redondant.
groupes_mois = panel.groupby('annee_mois')[caracteristiques_a_imputer]
panel[caracteristiques_a_imputer] = groupes_mois.transform(imputer_par_mediane_mois)
del groupes_mois
gc.collect()

colonnes_entierement_vides = []
for col in caracteristiques_a_imputer:
    if panel[col].isna().any():
        panel[col] = panel[col].fillna(panel[col].median())
    if panel[col].isna().any():
        colonnes_entierement_vides.append(col)
        panel[col] = panel[col].fillna(0)

print(f"Imputation terminee sur {len(caracteristiques_a_imputer)} caracteristiques (hors mvel1).")
if colonnes_entierement_vides:
    print()
    print("ATTENTION - colonnes entierement vides sur la population post-filtres (taille + liquidite) :")
    print(colonnes_entierement_vides)

total_manquant_chars = panel[caracteristiques].isna().sum().sum()
print(f"\nValeurs manquantes restantes dans les caracteristiques : {total_manquant_chars}")
assert total_manquant_chars == 0, "Il reste des valeurs manquantes parmi les caracteristiques !"
assert panel['mvel1'].isna().sum() == 0, "mvel1 ne devrait jamais avoir de valeurs manquantes ici !"


Imputation terminee sur 59 caracteristiques (hors mvel1).

Valeurs manquantes restantes dans les caracteristiques : 0


### B.4 Winsorizing des caractéristiques

Chaque caractéristique est cappée à ses 1er et 99e centiles, **calculés séparément pour chaque
mois**. Ce qui change, c'est **quand** : maintenant que les micro-caps et les titres illiquides
sont filtrés, le winsorizing s'applique à un panel déjà nettoyé de son bruit le plus
grossier — cohérent
avec l'ordre recommandé : filtrer d'abord *qui* on garde, nettoyer ensuite *les valeurs*
de ce qu'on garde.

In [23]:
def winsorize_groupe(serie, borne_bas=0.01, borne_haut=0.99):
    """Coupe les valeurs extremes d'une serie a ses percentiles bas/haut.
    Si la serie est entierement vide (tout NaN), on la renvoie telle quelle."""
    if serie.notna().sum() == 0:
        return serie
    lo = serie.quantile(borne_bas)
    hi = serie.quantile(borne_haut)
    return serie.clip(lo, hi)

# Meme optimisation qu'en B.3 : un seul groupby reutilise pour toutes les caracteristiques,
# au lieu d'un groupby('annee_mois') refait a chaque iteration de la boucle.
groupes_mois = panel.groupby('annee_mois')[caracteristiques]
panel[caracteristiques] = groupes_mois.transform(winsorize_groupe)
del groupes_mois
gc.collect()

print("Winsorizing termine (sur", len(caracteristiques), "caracteristiques).")


Winsorizing termine (sur 60 caracteristiques).


### B.5 Pas de découpage temporel fixe ici — fenêtres glissantes/extensives (notebooks 04 à 07)

Le projet utilise une succession de **fenêtres qui avancent dans le temps** (ré-entraînement
annuel, à la Gu, Kelly & Xiu 2020) plutôt qu'un seul découpage chronologique fixe train/
validation/test : soit *extensive* (le train grandit chaque année, sans jamais rien oublier),
soit *glissante* (le train garde une taille fixe et glisse). Comme chaque fenêtre a son propre
train/validation/test, la construction des fenêtres est isolée dans un fichier partagé,
`fenetres.py` (à la racine du projet, à côté de `config.py`), et **consommée directement par
les notebooks 04, 05, 06 et 07** plutôt qu'ici.

⚠️ **Pourquoi pas ici, dans le notebook 03 ?** Parce que la standardisation des variables macro
doit être recalculée **séparément pour le train de chaque fenêtre** (voir notebook 04,
section 1) — un seul calcul global ici recréerait exactement la fuite de données que ce
découpage cherche à éviter. Ce notebook s'arrête donc juste après le rank transform
ci-dessous : `panel_pret_modelisation.parquet` contient toutes les périodes, sans découpage ni
standardisation macro — les deux sont faits à la volée, fenêtre par fenêtre, dans les notebooks
suivants. Les paramètres des fenêtres (`TYPE_FENETRE`, `ANNEES_TRAIN_INITIAL`,
`ANNEES_VALIDATION`, `ANNEES_TEST_PAR_FENETRE`) sont dans `config.py`, comme tous les autres
paramètres partagés du projet.

### B.6 Transformation en rang des caractéristiques (rank transform)

Pour chaque caractéristique et chaque mois, on remplace la valeur brute par son **rang parmi
les entreprises de ce même mois**, ramené entre -1 (la plus faible valeur du mois) et +1 (la
plus forte). Ça rend les caractéristiques comparables entre elles (même échelle) et robustes
aux valeurs extrêmes, sans avoir besoin de connaître la distribution de chaque variable.

⚠️ Cette transformation utilise uniquement les entreprises **du même mois** — elle n'utilise
aucune information provenant d'un autre mois (passé ou futur), donc elle ne crée **aucune
fuite de données** entre train/validation/test. On peut donc l'appliquer avant le découpage
sans problème.

In [24]:
def rank_transform(serie):
    n = len(serie)
    if n <= 1:
        # un seul rang possible : on ne peut pas dire si l'entreprise est "haute" ou "basse"
        return pd.Series(0.0, index=serie.index)
    return 2 * (serie.rank(method='average') - 1) / (n - 1) - 1

# Meme optimisation qu'en B.3/B.4 : un seul groupby reutilise pour toutes les caracteristiques.
groupes_mois = panel.groupby('annee_mois')[caracteristiques]
panel[caracteristiques] = groupes_mois.transform(rank_transform)
del groupes_mois
gc.collect()

print("Transformation en rang terminee.")


Transformation en rang terminee.


In [25]:
# Diagnostic — a coller juste apres le rank transform
print("Ecart-type par caracteristique APRES rank transform (doit etre proche de 0.577 partout) :")
print(panel[caracteristiques].std().sort_values())

Ecart-type par caracteristique APRES rank transform (doit etre proche de 0.577 partout) :
sin              0.000000
divi             0.144707
divo             0.145029
rd               0.293050
convind          0.305148
securedind       0.477401
ps               0.550773
dy               0.552902
chinv            0.559068
tb               0.569039
operprof         0.569592
hire             0.570090
chpmia           0.570623
chempia          0.570677
pchgm_pchsale    0.570708
sgr              0.570720
depr             0.571142
cfp              0.571315
cfp_ia           0.571315
lgr              0.571327
chcsho           0.571344
gma              0.571352
egr              0.571455
agr              0.571496
tang             0.571835
age              0.572043
cashdebt         0.572290
quick            0.572540
herf             0.572707
currat           0.572786
salerec          0.572919
roic             0.573045
mom36m           0.573396
indmom           0.574019
cashpr           0.574124


In [26]:
# Verification : toutes les valeurs doivent etre entre -1 et 1
bornes = panel[caracteristiques].agg(['min', 'max']).T
print("Min/Max apres transformation (doit etre entre -1 et 1) :")
bornes


Min/Max apres transformation (doit etre entre -1 et 1) :


,min,max
mvel1,-0.990338,0.990338
beta,-0.990338,0.990333
betasq,-0.990338,0.990333
chmom,-0.990338,0.990338
dolvol,-0.990338,0.990338
idiovol,-0.990338,0.990338
indmom,-0.990220,0.990161
mom1m,-0.990338,0.990338
mom6m,-0.990338,0.990338
mom12m,-0.990338,0.990338


### Variables macro : pas de standardisation ici non plus

Contrairement aux caractéristiques d'entreprise (rank-transformées ci-dessus, sans fuite
possible), les variables macro ont **une seule valeur par mois** (identique pour toutes les
entreprises) et se standardisent classiquement en z-score : `(valeur - moyenne) / ecart-type`.
Mais cette moyenne et cet écart-type doivent être calculés **sur le train**, et — comme
expliqué en B.5 — le train n'est plus unique : il change à chaque fenêtre. On sauvegarde donc
les variables macro **brutes** (non standardisées) ci-dessous ; c'est `fenetres.py` (fonction
`preparer_fenetre`) qui recalcule cette standardisation à la volée pour chaque fenêtre, dans
les notebooks 04, 05 et 06.

### B.7 Vérifications finales et sauvegarde

In [27]:
print("Dimensions finales :", panel.shape)
print("Periode couverte :", panel['annee_mois'].min(), "a", panel['annee_mois'].max())
print("Nombre de mois distincts :", panel['annee_mois'].nunique())


Dimensions finales : (2615807, 76)
Periode couverte : 198002 a 202112
Nombre de mois distincts : 503


In [28]:
print("Valeurs manquantes restantes (doit etre 0) :")
manquants = panel[caracteristiques + macro_predicteurs + [cible]].isna().sum()
print(manquants[manquants > 0] if manquants.sum() > 0 else "Aucune valeur manquante.")


Valeurs manquantes restantes (doit etre 0) :
Aucune valeur manquante.


In [29]:
# On lit le dtype d'origine directement depuis les categories elles-memes (pas besoin de l'avoir
# garde en memoire depuis A.6bis) : ca marche donc aussi si la partie B a ete relancee seule,
# apres un redemarrage du kernel, en rechargeant panel_final.parquet depuis le disque.
# But : panel_pret_modelisation.parquet est le fichier que lisent les notebooks 04 a 08, qui ne
# savent rien de la conversion en 'category' faite en A.6bis (elle sert seulement a alleger la
# memoire PENDANT ce notebook) -- on revient donc au dtype d'origine juste avant de sauvegarder,
# pour ne rien changer au comportement des notebooks suivants (fenetres.py, merges, graphiques...).
for col in ['annee_mois', 'permno']:
    if isinstance(panel[col].dtype, pd.CategoricalDtype):
        panel[col] = panel[col].astype(panel[col].cat.categories.dtype)

print("Types juste avant sauvegarde (ne doivent plus etre 'category') :")
print(panel[['permno', 'annee_mois']].dtypes)

panel.to_parquet(config.FICHIER_PANEL_MODELISATION, index=False)
print("Fichier sauvegarde :", config.FICHIER_PANEL_MODELISATION)


Types juste avant sauvegarde (ne doivent plus etre 'category') :
permno        int64
annee_mois      str
dtype: object
Fichier sauvegarde : C:\Users\aless\OneDrive\Documents\Pantheon Sorbonne\cour\Memoire\V11\data\processed\panel_pret_modelisation.parquet


**Résumé de la partie B :** univers investissable filtré mois par mois (taille puis
liquidité), caractéristiques retenues (notebook 02, section A.3bis) imputées par médiane du mois
(sur la population post-filtre), caractéristiques winsorisées (1er/99e centile par mois) puis transformées en
rangs dans `[-1, 1]`. **Pas de découpage temporel ni de standardisation macro dans ce
notebook** : les deux dépendent de la fenêtre (extensive ou glissante) en cours, et
sont donc calculés à la volée par `fenetres.py`, réutilisé par les notebooks 04 à 07 — voir
B.5 ci-dessus. Résultat sauvegardé dans `data/processed/panel_pret_modelisation.parquet` (toutes
les périodes, sans colonne `split`) — plus aucune valeur manquante dans les caractéristiques,
les prédicteurs macro, ni la cible.

## Résumé global et prochaines étapes

- **Partie A** : panel fusionné (caractéristiques + rendements + macro décalée + cible
  `excess_return`), sauvegardé dans `data/processed/panel_final.parquet`.
- **Partie B**, directement enchaînée sur le panel en mémoire : univers filtré, valeurs
  manquantes imputées, caractéristiques winsorisées et rang-transformées — sauvegardé dans
  `data/processed/panel_pret_modelisation.parquet`, **sans découpage temporel ni standardisation
  macro** (toutes les périodes y sont mélangées, avec `annee_mois` pour les distinguer).

**Prochaine étape (Notebook 04) :** entraîner le premier modèle du projet — une régression
linéaire simple (OLS, sans hyperparamètre), qui sert de plancher de comparaison pour l'Elastic
Net (notebook 05) et LightGBM (notebook 06). Chaque modèle est ré-entraîné plusieurs fois sur
des **fenêtres glissantes/extensives** construites par `fenetres.py` — voir sa section 1.